# version5 — training ladder S1–S7 (AGENT_PLAN_V5)

**Moi lan chay = 1 rung (`RUN_ID`), toi da ~10 h** (`--time_budget_hours 10`): script dung
sau epoch da hoan tat, luu `resume_state.pth`, nen khong bao gio cham tran 12 h cua Kaggle.
Chia `epochs` thanh nhieu `STAGE` neu can (`END_EPOCH` + `RESUME_FROM`).

| rung | thay doi so voi rung truoc | muc dich |
|---|---|---|
| **S1** | DINOv2+LoRA, head CLS, CE sach, `label_smoothing=0`, khong LIB/GSR/DCS, khong AIGB | baseline `p(z=logit)` |
| **S2** | + online aug `bfree` (placeholder Appendix A) | do nhay voi aug |
| **S2b** | + aug `light` / `strong` | kiem tra do nhay |
| **S3** | + AIGB 25% (`other_frac=0.25`), vanilla | so voi ours-v2 (khac head/ls) |
| **S4** | full FT — **KHONG chay tu notebook nay** | can audit `train_full_ft.py` truoc |
| **S5** | S3 + LIB/GSR `zero_init`, `gsr_window=7`, head `cls_plus_patch` (`gamma=0`) | buoc 0 ≡ S3 |
| **S6** | S3 + `dcs_mode=class_aware` (view 2 = lifecycle degrade) | DCS khong keo real |
| **S7** | rung tot nhat (`S7_FROM`) × 6 epoch, 3 seed | ket qua cuoi |

Quy uoc: moi rung chi doi MOT nhom bien so; moi thu khac giu nguyen YAML repo.
So sanh voi so lieu v1/v2/v3 chi sau khi doc `research/eval-v2-proposal.md`.

**Inputs** (attach truoc khi chay): `xla-setup` (wheels + `bfree_src` + DINOv2),
`b-free-training-data`, `ai-genbench-v1` (chi can cho S3+).

## Rung S1–S7 — doc truoc khi sua cell RECIPE

- `RUN_ID` phai la mot key trong `RECIPES`; `S7` lay `S7_FROM`.
- `RECIPES` chi chua cac bien **khac nhau giua cac rung**; bien khac doc tu YAML repo
  (`configs/bfree_dcs.yaml`) de khong hardcode.
- `label_smoothing = 0.0` cho S1-S7 (khop BCE cua B-Free) duoc ghi vao `run_config.yaml`
  cua tung run, KHONG truyen tren CLI — cung voi `head_mode/use_*/gsr_*/lib_*/dcs_*`,
  `paired/other_frac/val_aug`, `online_aug`. CLI chi con lr/lora/val-split/worker/seed.
- **S4 bi chan**: `train_lora.py` khong ho tro full fine-tune; phai audit
  `train_full_ft.py` theo cung pipeline (paired + aug + val_aug) roi moi chay.
- Nhanh `integration/loss-backbone` phai duoc push truoc khi tao bundle `bfree_src`.

## Stage / resume

- `END_EPOCH` = epoch cuoi cua phien nay; `RESUME_FROM = None` o stage 1,
  `.../resume_state.pth` o stage sau.
- `--epochs` KHONG doi giua cac stage (lich cosine tinh tren tong so epoch).
- **Tham chieu do thuc te (notebook v3, metadata):** stage 1 = 4 epoch trong 21.123 s
  (5.281 s/epoch ≈ 1,47 h/epoch, AIGB bat, batch auto, 504 px). Do la moc duy nhat da do.
  Paired mode xu ly 2 anh/stem moi epoch (real + fake) thay vi 1 nhu v3, nen **do epoch 1
  cua tung rung roi moi chon `END_EPOCH`** — dung doan. Xem dong
  `--> epoch k/N done` trong `train_log.txt` de lay so giay that; `--time_budget_hours 10`
  tu dung sau epoch hoan tat neu vuot.
- Ckpt cua stage trung gian KHONG dung de eval: `bfree_globalforge_lora_r16.pth` o do la
  epoch `END_EPOCH`, chi ckpt cua stage cuoi (`END_EPOCH == N_EPOCHS`) moi la ket qua rung.

In [1]:
import logging
import os
import sys
from pathlib import Path

logger = logging.getLogger("bfree")
# DEBUG ở logger để file handler (cell sau) bắt cả step-level log; console handler
# riêng lọc lại INFO. propagate=False để log không bubble lên root logger (tránh
# in trùng 2 lần khi transformers/huggingface_hub tự basicConfig lúc import).
logger.setLevel(logging.DEBUG)
logger.propagate = False
logger.handlers.clear()
_console = logging.StreamHandler()
_console.setLevel(logging.INFO)
_console.setFormatter(logging.Formatter(fmt="%(asctime)s | %(levelname)-7s | %(message)s", datefmt="%H:%M:%S"))
logger.addHandler(_console)

logger.info("stdlib + logging ready")

16:29:49 | INFO    | stdlib + logging ready


In [2]:
import glob
from pathlib import Path

WORK = Path("/kaggle/working")


def require_dir(p, what):
    p = Path(p)
    if not p.is_dir():
        raise FileNotFoundError(
            f"NOT FOUND (thu muc): {what}\n  mong doi: {p}\n"
            "  -> Attach dung input, hoac sua duong dan o cell CONFIG-PATHS.")
    return p


def require_file(p, what):
    p = Path(p)
    if not p.is_file():
        raise FileNotFoundError(
            f"NOT FOUND (file): {what}\n  mong doi: {p}\n"
            "  -> Attach dung input, hoac sua duong dan o cell CONFIG-PATHS.")
    return p


def unwrap_dir(path):
    """Bo 1 tang long thua: <dir>/<dir> -> <dir> (neu ton tai)."""
    path = Path(path)
    child = path / path.name
    return child if child.is_dir() else path


# ============================================================
# CONFIG-PATHS -- TAT CA duong dan viet thang, KHONG glob / KHONG doan.
# ============================================================

WHEELS_DIR = Path("/kaggle/input/notebooks/baopv051/xla-setup/wheels_rtxpro6000")
REPO_SRC   = Path("/kaggle/input/notebooks/baopv051/xla-setup/bfree_src")
DINOV2_SD  = Path("/kaggle/input/notebooks/baopv051/xla-setup/models/vit_base_patch14_reg4_dinov2/model.safetensors")
TRAIN_DATA_ROOT = Path("/kaggle/input/datasets/baopv051/b-free-training-data")
AI_GENBENCH_DIR = Path("/kaggle/input/datasets/baopv051/ai-genbench-v1")

require_dir(WHEELS_DIR, "wheel bundle 'wheels_rtxpro6000' (output cua notebook 01)")
require_dir(REPO_SRC / "code" / "networks", "B-Free repo source 'bfree_src' (output cua notebook 01)")
require_file(DINOV2_SD, "DINOv2 ViT-B/14 reg4 weights (output cua notebook 01)")
require_dir(unwrap_dir(TRAIN_DATA_ROOT / "COCO_real_512"), "COCO_real_512 (B-Free training data)")
AIGB_OK = AI_GENBENCH_DIR.is_dir()   # chi can cho S3+ (other_frac > 0)

wheels = sorted(glob.glob(str(WHEELS_DIR / "*.whl")))
assert wheels, f"No .whl files inside {WHEELS_DIR}"
logger.info(f"WHEELS_DIR       = {WHEELS_DIR} ({len(wheels)} wheels)")
logger.info(f"REPO_SRC         = {REPO_SRC}")
logger.info(f"DINOV2_SD        = {DINOV2_SD}")
logger.info(f"TRAIN_DATA_ROOT  = {TRAIN_DATA_ROOT}")
logger.info(f"AI_GENBENCH_DIR  = {AI_GENBENCH_DIR}")

WORK.mkdir(parents=True, exist_ok=True)
_fh = logging.FileHandler(str(WORK / "train_log.txt"), mode="a", encoding="utf-8")
_fh.setLevel(logging.DEBUG)  # file giữ cả step-level debug log
_fh.setFormatter(logging.Formatter(fmt="%(asctime)s | %(levelname)-7s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S"))
logger.addHandler(_fh)

16:29:49 | INFO    | WHEELS_DIR       = /kaggle/input/notebooks/baopv051/xla-setup/wheels_rtxpro6000 (40 wheels)
16:29:49 | INFO    | REPO_SRC         = /kaggle/input/notebooks/baopv051/xla-setup/bfree_src
16:29:49 | INFO    | DINOV2_SD        = /kaggle/input/notebooks/baopv051/xla-setup/models/vit_base_patch14_reg4_dinov2/model.safetensors
16:29:49 | INFO    | TRAIN_DATA_ROOT  = /kaggle/input/datasets/baopv051/b-free-training-data
16:29:49 | INFO    | AI_GENBENCH_DIR  = /kaggle/input/datasets/baopv051/ai-genbench-v1


In [3]:
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONHASHSEED"] = "0"
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"

logger.info("offline env vars set (HF hub blocked, bytecode off — repo source stays read-only in input)")

16:29:49 | INFO    | offline env vars set (HF hub blocked, bytecode off — repo source stays read-only in input)


In [4]:
# Pin EXACT versions moi package (== vi du image Kaggle da co torch 2.10/
# transformers 5.0/peft 0.19 — spec long se lam pip no-op). --no-deps de
# khong resolve ra dep moi (vd hf-xet) — bundle da chua day du leaf deps.
!pip install --no-index --no-deps --find-links="{WHEELS_DIR}" \
    torch==2.8.0 torchvision==0.23.0 timm==1.0.22 peft==0.15.2 \
    transformers==4.55.4 huggingface_hub==0.36.0 tokenizers==0.21.1 \
    accelerate==1.3.0 pandas==2.3.3 safetensors numpy matplotlib seaborn \
    scikit-learn scipy pyyaml pillow tqdm requests filelock fsspec \
    packaging typing-extensions regex certifi charset-normalizer idna urllib3 \
    psutil joblib threadpoolctl python-dateutil six pytz tzdata \
    contourpy cycler fonttools kiwisolver pyparsing

import torch
import timm
import peft
import transformers
import huggingface_hub
import tokenizers
import accelerate

assert torch.__version__.startswith("2.8.0"), f"torch {torch.__version__} != 2.8.0+cu128 (bundle wrong?)"
assert timm.__version__ == "1.0.22", f"timm {timm.__version__} != 1.0.22"
assert transformers.__version__ == "4.55.4", f"transformers {transformers.__version__} != 4.55.4"
assert peft.__version__ == "0.15.2", f"peft {peft.__version__} != 0.15.2"
assert huggingface_hub.__version__ == "0.36.0", f"huggingface_hub {huggingface_hub.__version__} != 0.36.0 (transformers 4.55.4 needs hub<1.0)"
assert tokenizers.__version__.startswith("0.21."), f"tokenizers {tokenizers.__version__} != 0.21.x (transformers 4.55.4 needs tokenizers<0.22)"
assert accelerate.__version__ == "1.3.0", f"accelerate {accelerate.__version__} != 1.3.0"
import pandas
assert pandas.__version__.split(".")[0] == "2", "pandas>=3 breaks sklearn (risk table)"
logger.info(f"pip --no-index OK | torch={torch.__version__} timm={timm.__version__} "
            f"peft={peft.__version__} transformers={transformers.__version__} "
            f"accelerate={accelerate.__version__}")

Looking in links: /kaggle/input/notebooks/baopv051/xla-setup/wheels_rtxpro6000
Processing /kaggle/input/notebooks/baopv051/xla-setup/wheels_rtxpro6000/torch-2.8.0+cu128-cp312-cp312-manylinux_2_28_x86_64.whl
Processing /kaggle/input/notebooks/baopv051/xla-setup/wheels_rtxpro6000/torchvision-0.23.0+cu128-cp312-cp312-manylinux_2_28_x86_64.whl
Processing /kaggle/input/notebooks/baopv051/xla-setup/wheels_rtxpro6000/timm-1.0.22-py3-none-any.whl
Processing /kaggle/input/notebooks/baopv051/xla-setup/wheels_rtxpro6000/peft-0.15.2-py3-none-any.whl
Processing /kaggle/input/notebooks/baopv051/xla-setup/wheels_rtxpro6000/transformers-4.55.4-py3-none-any.whl
Processing /kaggle/input/notebooks/baopv051/xla-setup/wheels_rtxpro6000/huggingface_hub-0.36.0-py3-none-any.whl
Processing /kaggle/input/notebooks/baopv051/xla-setup/wheels_rtxpro6000/tokenizers-0.21.1-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
Processing /kaggle/input/notebooks/baopv051/xla-setup/wheels_rtxpro6000/accelerate-1.3.0

16:30:45 | INFO    | pip --no-index OK | torch=2.8.0+cu128 timm=1.0.22 peft=0.15.2 transformers=4.55.4 accelerate=1.3.0


In [5]:
import sys

sys.path.insert(0, str(Path(REPO_SRC) / "code"))

stubs = sorted(glob.glob(str(Path(REPO_SRC) / "code" / "modules" / "*_stub.py")))
assert not stubs, f"Stub files still present (K0 not merged?): {stubs}"

import train_lora  # noqa: F401  — train-only script (orchestration), chay qua accelerate o cell sau

logger.info(f"repo import OK | train_lora tai {train_lora.__file__}")

16:30:46 | INFO    | repo import OK | train_lora tai /kaggle/input/notebooks/baopv051/xla-setup/bfree_src/code/train_lora.py


In [6]:
# ============================================================
# RECIPE -- chon rung (S1..S7). Cac bien khac lay tu YAML repo.
# ============================================================
import yaml

RUN_ID = "S7"                  # S1 | S2 | S2b-light | S2b-strong | S3 | S5 | S6 | S7
SEED = 42
S7_FROM = "S3"                 # chi dung khi RUN_ID == "S7": rung duoc promote

assert RUN_ID != "S4", ("S4 (full FT) chua co trong train_lora.py: phai audit train_full_ft.py theo pipeline v4 truoc khi chay.")

_BASE = {
    "S1":  dict(head_mode="cls", use_lib=False, use_gsr=False, use_dcs=False,
                online_aug=None, other_frac=0.0, epochs=2),
    "S2":  dict(head_mode="cls", use_lib=False, use_gsr=False, use_dcs=False,
                online_aug="bfree", other_frac=0.0, epochs=2),
    "S2b-light": dict(head_mode="cls", use_lib=False, use_gsr=False, use_dcs=False,
                      online_aug="light", other_frac=0.0, epochs=2),
    "S2b-strong": dict(head_mode="cls", use_lib=False, use_gsr=False, use_dcs=False,
                       online_aug="strong", other_frac=0.0, epochs=2),
    "S3":  dict(head_mode="cls", use_lib=False, use_gsr=False, use_dcs=False,
                online_aug="bfree", other_frac=0.25, epochs=2),
    "S5":  dict(head_mode="cls_plus_patch", use_lib=True, use_gsr=True, use_dcs=False,
                gsr_zero_init=True, gsr_window=7, lib_beta_init=-10.0,
                online_aug="bfree", other_frac=0.25, epochs=2),
    "S6":  dict(head_mode="cls", use_lib=False, use_gsr=False, use_dcs=True,
                dcs_mode="class_aware",
                online_aug="bfree", other_frac=0.25, epochs=2),
}
if RUN_ID == "S7":
    assert S7_FROM in _BASE, f"S7_FROM={S7_FROM} khong co trong _BASE"
    RECIPE = {**_BASE[S7_FROM], "epochs": 6}
else:
    RECIPE = _BASE[RUN_ID]

cfg = yaml.safe_load(open(Path(REPO_SRC) / "code" / "configs" / "bfree_dcs.yaml", encoding="utf-8"))
cfg["backbone"].update({
    "use_lib": RECIPE["use_lib"],
    "use_gsr": RECIPE["use_gsr"],
    "head_mode": RECIPE["head_mode"],
    "gsr_window": RECIPE.get("gsr_window", cfg["backbone"]["gsr_window"]),
    "gsr_zero_init": RECIPE.get("gsr_zero_init", False),
    "lib_beta_init": RECIPE.get("lib_beta_init", cfg["backbone"]["lib_beta_init"]),
})
cfg["loss"].update({
    "use_dcs": RECIPE["use_dcs"],
    "dcs_mode": RECIPE.get("dcs_mode", cfg["loss"]["dcs_mode"]),
    "label_smoothing": 0.0,   # S1-S7: khop BCE cua B-Free (plan 1.3)
})
cfg["data"].update({"paired": True, "other_frac": RECIPE["other_frac"], "val_aug": True})
cfg["online_aug"]["enabled"] = RECIPE["online_aug"] is not None
if RECIPE["online_aug"]:
    cfg["online_aug"]["preset"] = RECIPE["online_aug"]
cfg["multi_severity"]["enabled"] = False   # 1 view (DCS S6 dung lifecycle-degraded view)

OUTPUT_DIR = WORK / f"{RUN_ID}_seed{SEED}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_CONFIG = OUTPUT_DIR / "run_config.yaml"
yaml.safe_dump(cfg, open(RUN_CONFIG, "w", encoding="utf-8"), sort_keys=False)

N_EPOCHS = int(RECIPE["epochs"])
logger.info(f"RECIPE {RUN_ID}: {RECIPE}")
logger.info(f"config -> {RUN_CONFIG} | output -> {OUTPUT_DIR}")

16:30:46 | INFO    | RECIPE S7: {'head_mode': 'cls', 'use_lib': False, 'use_gsr': False, 'use_dcs': False, 'online_aug': 'bfree', 'other_frac': 0.25, 'epochs': 6}
16:30:46 | INFO    | config -> /kaggle/working/S7_seed42/run_config.yaml | output -> /kaggle/working/S7_seed42


In [7]:
# ============================================================
# STAGE -- phien nay train toi epoch nao. --epochs = N_EPOCHS o moi stage.
# ============================================================
import yaml

_cfg = yaml.safe_load(open(RUN_CONFIG, encoding="utf-8"))
END_EPOCH = 6                   # stage 1; stage sau: 4, 6, ...
RESUME_FROM = None              # stage sau: OUTPUT_DIR cua stage truoc / "resume_state.pth"
TIME_BUDGET_HOURS = 10.0        # dung sau epoch, truoc tran 12 h

assert 1 <= END_EPOCH <= N_EPOCHS, (END_EPOCH, N_EPOCHS)
if RESUME_FROM is not None:
    require_file(RESUME_FROM, "resume_state.pth cua stage truoc")
logger.info(f"STAGE: {RUN_ID} -> epoch {END_EPOCH}/{N_EPOCHS} | resume_from={RESUME_FROM} "
            f"| budget={TIME_BUDGET_HOURS}h")
if END_EPOCH < N_EPOCHS:
    _rel = f"{RUN_ID}_seed{SEED}/resume_state.pth"
    logger.warning(f"stage sau: RESUME_FROM='/kaggle/input/notebooks/baopv051/"
                   f"<slug-notebook-nay>/{_rel}'  (ngoac: END_EPOCH={N_EPOCHS})")
    logger.warning(f"KHONG eval ckpt cua stage trung gian: "
                   f"bfree_globalforge_lora_r16.pth o day la epoch {END_EPOCH}, khong phai epoch cuoi.")

16:30:46 | INFO    | STAGE: S7 -> epoch 6/6 | resume_from=None | budget=10.0h


In [8]:
# ============================================================
# TRAINING -- subprocess de bat exit code; cwd = code dir.
# Progress bar tqdm -> stderr, khong sinh ipywidgets state.
# ============================================================
import subprocess
import sys

cmd = [
    "accelerate", "launch", "--num_processes", "1", "--num_machines", "1",
    "--mixed_precision", "bf16", "--dynamo_backend", "no",
    "train_lora.py",
    "--train_data_root", str(TRAIN_DATA_ROOT),
    "--dinov2_sd", str(DINOV2_SD),
    "--output_dir", str(OUTPUT_DIR),
    "--config", str(RUN_CONFIG),
    "--epochs", str(N_EPOCHS), "--end_epoch", str(END_EPOCH),
    "--batch_size", "0",
    "--lr", "1e-4", "--weight_decay", "1e-4", "--max_grad_norm", "1.0",
    "--scheduler_eta_min", "1e-7",
    "--lora_rank", "16", "--lora_alpha", "32", "--lora_dropout", "0.05",
    "--val_md5_percentile", "3", "--val_fake_variant", "SD2.1_selfconditioned",
    "--pair_real_prob", "0.5",
    "--degrade_max_rounds", "4", "--degrade_round_probs", "0.15,0.30,0.25,0.20,0.10",
    "--extra_source_val_percentile", "30",
    "--num_workers", "4", "--seed", str(SEED),
    "--select", "last",
    "--time_budget_hours", str(TIME_BUDGET_HOURS),
]
if RECIPE["other_frac"] > 0:
    cmd += ["--ai_genbench_dir", str(AI_GENBENCH_DIR),
            "--aigenbench_split", "train",
            "--aigenbench_max_per_gen", "4000",
            "--aigenbench_val_frac", "0.1"]
if RESUME_FROM is not None:
    cmd += ["--resume_from", str(RESUME_FROM)]

logger.info("cmd = " + " ".join(cmd))
r = subprocess.run(cmd, cwd=str(Path(REPO_SRC) / "code"), text=True,
                   encoding="utf-8", errors="replace")
assert r.returncode == 0, f"train_lora.py exit {r.returncode} — xem log phia tren"


16:30:46 | INFO    | cmd = accelerate launch --num_processes 1 --num_machines 1 --mixed_precision bf16 --dynamo_backend no train_lora.py --train_data_root /kaggle/input/datasets/baopv051/b-free-training-data --dinov2_sd /kaggle/input/notebooks/baopv051/xla-setup/models/vit_base_patch14_reg4_dinov2/model.safetensors --output_dir /kaggle/working/S7_seed42 --config /kaggle/working/S7_seed42/run_config.yaml --epochs 6 --end_epoch 6 --batch_size 0 --lr 1e-4 --weight_decay 1e-4 --max_grad_norm 1.0 --scheduler_eta_min 1e-7 --lora_rank 16 --lora_alpha 32 --lora_dropout 0.05 --val_md5_percentile 3 --val_fake_variant SD2.1_selfconditioned --pair_real_prob 0.5 --degrade_max_rounds 4 --degrade_round_probs 0.15,0.30,0.25,0.20,0.10 --extra_source_val_percentile 30 --num_workers 4 --seed 42 --select last --time_budget_hours 10.0 --ai_genbench_dir /kaggle/input/datasets/baopv051/ai-genbench-v1 --aigenbench_split train --aigenbench_max_per_gen 4000 --aigenbench_val_frac 0.1
16:30:53 | INFO    | [multi_

normalize RESNET
normalize RESNET
normalize RESNET
normalize RESNET
normalize RESNET
normalize RESNET
normalize RESNET


16:51:02 | INFO    | [*] best checkpoint saved (epoch 1, bAcc 0.9404)
16:51:03 | INFO    | --> epoch 1/6 done | best bAcc so far = 0.9404
17:10:39 | INFO    | [*] best checkpoint saved (epoch 2, bAcc 0.9463)
17:10:39 | INFO    | --> epoch 2/6 done | best bAcc so far = 0.9463
17:30:10 | INFO    | [*] best checkpoint saved (epoch 3, bAcc 0.9613)
17:30:10 | INFO    | --> epoch 3/6 done | best bAcc so far = 0.9613
17:49:41 | INFO    | --> epoch 4/6 done | best bAcc so far = 0.9613
18:09:10 | INFO    | [*] best checkpoint saved (epoch 5, bAcc 0.9672)
18:09:11 | INFO    | --> epoch 5/6 done | best bAcc so far = 0.9672
18:28:41 | INFO    | [*] best checkpoint saved (epoch 6, bAcc 0.9686)
18:28:41 | INFO    | --> epoch 6/6 done | best bAcc so far = 0.9686
18:28:42 | INFO    | selected checkpoint for eval (--select last): bfree_globalforge_lora_r16.pth
18:28:42 | INFO    | saved training_curves.png
18:28:42 | INFO    | TRAINING COMPLETE


In [9]:
# ============================================================
# KIEM TRA KET QUA + PROVENANCE (dung cho harness v4)
# ============================================================
import hashlib
import json
import subprocess

import pandas as pd
from IPython.display import display

csv_path = OUTPUT_DIR / "train_log.csv"
ckpt_path = OUTPUT_DIR / "bfree_globalforge_lora_r16.pth"
assert csv_path.is_file(), csv_path
assert ckpt_path.is_file(), ckpt_path

df = pd.read_csv(csv_path)
assert df["train_loss"].notna().all() and df["val_loss"].notna().all(), df.tail()
assert len(df) >= 1
if RECIPE["head_mode"] == "cls_plus_patch":
    assert df["gamma"].notna().all(), "gamma phai duoc log cho head cls_plus_patch"

last = df.iloc[-1]
logger.info(f"{RUN_ID} seed={SEED}: {len(df)} epoch(s) | best bAcc={df['val_bacc'].max():.4f} "
            f"| last total={last['train_loss']:.4f} val_auc={last['val_auc']:.4f} "
            f"val_bacc={last['val_bacc']:.4f} gamma={last['gamma']}")

def _git_sha():
    try:
        return subprocess.check_output(["git", "-C", str(REPO_SRC), "rev-parse", "HEAD"],
                                       text=True).strip()
    except Exception:
        return "unknown"

meta = {
    "run_id": RUN_ID, "seed": SEED, "recipe": RECIPE,
    "epochs_done": int(last["epoch"]), "epochs_total": N_EPOCHS,
    "git_sha": _git_sha(),
    "ckpt_sha256": hashlib.sha256(ckpt_path.read_bytes()).hexdigest(),
    "output_dir": str(OUTPUT_DIR),
}
(OUTPUT_DIR / "run_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
logger.info(f"run_meta.json -> {OUTPUT_DIR / 'run_meta.json'}")
display(df.tail(8))

18:28:48 | INFO    | S7 seed=42: 6 epoch(s) | best bAcc=0.9686 | last total=0.0808 val_auc=0.9970 val_bacc=0.9686 gamma=nan
fatal: not a git repository (or any parent up to mount point /kaggle/input/notebooks/baopv051)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
18:28:48 | INFO    | run_meta.json -> /kaggle/working/S7_seed42/run_meta.json


,epoch,train_loss,train_ce,train_dcs,val_loss,val_auc,val_bacc,gamma
0,1,0.2367,0.2367,0.0,0.1583,0.9867,0.9404,NaN
1,2,0.1416,0.1416,0.0,0.1249,0.9917,0.9463,NaN
2,3,0.1144,0.1144,0.0,0.1019,0.9943,0.9613,NaN
3,4,0.0975,0.0975,0.0,0.0944,0.9960,0.9599,NaN
4,5,0.0871,0.0871,0.0,0.0798,0.9967,0.9672,NaN
5,6,0.0808,0.0808,0.0,0.0746,0.9970,0.9686,NaN


## Sau khi chay

1. Tai `OUTPUT_DIR` xuong, attach lam Input cho harness v4; trong harness dat
   `OURS_CKPT = .../<RUN_ID>_seed<SEED>/bfree_globalforge_lora_r16.pth`.
2. So sanh **trong cung protocol v2** (`research/eval-v2-proposal.md`); S1 cac so cu.
3. Ghi ket qua vao `research/log.md` + `research/directions/D-recipe-v5/README.md`
   (gate S1–S7 da dang ky truoc).
4. `run_meta.json` (git sha + ckpt sha256) phai khop voi `provenance` cua harness;
   lech sha => ket qua khong dung de so sanh.

**Luu y:** `--time_budget_hours` chi dung sau khi mot epoch xong. Epoch 0 (~44 min o 504 px)
khong bi cat giua chung, nen 10 h van an toan trong gioi han 12 h.